In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from collections import defaultdict
from matplotlib.backends.backend_pdf import PdfPages

data = pd.read_csv(
    "data.csv",
    sep=";",
    encoding="utf-8-sig",   # -sig supprime le BOM proprement
    decimal=",",
    low_memory=False
)

print(data.shape)   # (4271, 256)
print(data.dtypes.value_counts())

(4271, 256)
float64    245
str          8
int64        3
Name: count, dtype: int64


In [2]:
### Paramètres globaux

def chips_df(df:pd.DataFrame):
    chip_dict={}
    LEN=defaultdict(int)
    for chip in df.Chip_Type:
        chip_dict[chip]=df[df.Chip_Type == chip]
        LEN[chip]=len(chip_dict[chip])
    return chip_dict,LEN
COLUMNS=list(data.columns)
chip_dict,LEN=chips_df(data)
CHIPS=list(data.Chip_Type.unique())

In [39]:
print(CHIPS)
KEY="ISAC_V2"

['ISAC_V1', 'ALEX', 'ISAC_V2']


In [40]:
def one_hot(df : pd.DataFrame,col : str)->pd.DataFrame:
    # 1. Nettoyer les espaces éventuels autour des virgules puis faire le one-hot encoding
    one_hot = (
        df[col]
        .str.replace(r"\s*,\s*", ",", regex=True)
        .str.get_dummies(sep=",")
    )
    # 2. (Optionnel) Trier les colonnes dans l'ordre numérique : '0', '1', '2', ..., '10'
    one_hot = one_hot.reindex(sorted(one_hot.columns, key=int), axis=1)
    # 3. (Optionnel) Convertir les 0/1 en True/False (booléens) et ajouter un préfixe
    one_hot = one_hot.add_prefix("cofacteur_")
    # 4. Rattacher au DataFrame principal et supprimer l'ancienne colonne
    df = pd.concat([df.drop(columns=[col]), one_hot], axis=1)
    return df


In [41]:
def cleaning(chip):
    # création des variables pour cette chips
    df=chip_dict[chip]
    N=LEN["ISAC_V1"]
    TTINT = ["Treatment_of_athsma", "Age_of_onsets", "Treatment_of_atopic_dematitis"] # Turn To INT
    for col in TTINT:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("Int64")
    df=one_hot(df,"General_cofactors")      #C'est une liste de numéro de cofacteur qui sont présent
    #Nettoyage
    df = df.replace(9, np.nan)
    dico_nan={col : round(df[col].isna().sum()/N*100,2) for col in df.columns}
    COL_TO_SUPP={col for col in  df.columns if dico_nan[col]>20}
    df.drop(COL_TO_SUPP, axis=1,inplace=True)
    return df


In [42]:
df=cleaning(KEY)

C:\Users\victo\AppData\Local\Temp\ipykernel_16912\443990099.py:13: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(COL_TO_SUPP, axis=1,inplace=True)


In [43]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 781 entries, 526 to 3757
Columns: 135 entries, Patient_ID to cofacteur_11
dtypes: Int64(3), float64(117), int64(11), str(4)
memory usage: 832.1 KB
None


In [44]:
# Bon maintenant on va virer les variables nulles
STR_COL = df.select_dtypes(include=["object", "string"]).columns.tolist()
df.drop(STR_COL, axis=1,inplace=True)

C:\Users\victo\AppData\Local\Temp\ipykernel_16912\1219955111.py:3: Pandas4Warning: The inplace keyword in DataFrame.drop is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  df.drop(STR_COL, axis=1,inplace=True)


In [45]:
print(STR_COL)

['Patient_ID', 'Chip_Type', 'French_Residence_Department', 'French_Region']


In [46]:
print(df.info())

<class 'pandas.DataFrame'>
Index: 781 entries, 526 to 3757
Columns: 131 entries, Age to cofacteur_11
dtypes: Int64(3), float64(117), int64(11)
memory usage: 807.7 KB
None


In [47]:
"""
Okkay donc faut que je vire l'ID patient et le CHIP TYPE
Treatment_of_athsma : en vrai c'est un int faut changer ca
Age_of_onsets : en vrai c'est un int faut changer ca
Treatment_of_atopic_dematitis : en vrai c'est un int faut changer ca
General_cofactors : c'est la liste de cofacteur. C'est des une liste faut trouver un moyen de coder ca. Je pense on va faire une variable par cofacteur
"""

"\nOkkay donc faut que je vire l'ID patient et le CHIP TYPE\nTreatment_of_athsma : en vrai c'est un int faut changer ca\nAge_of_onsets : en vrai c'est un int faut changer ca\nTreatment_of_atopic_dematitis : en vrai c'est un int faut changer ca\nGeneral_cofactors : c'est la liste de cofacteur. C'est des une liste faut trouver un moyen de coder ca. Je pense on va faire une variable par cofacteur\n"

In [48]:
def plot_matrix(df):
    # 1. Sélection des colonnes numériques
    df_num = df.select_dtypes(include=np.number)

    # 2. Matrice de corrélation
    corr = df_num.corr(method="spearman")      # ou "spearman" (cf. remarques)

    # 3. Heatmap hiérarchisée (clustering des colonnes pour faire émerger les familles)
    # plt.figure(figsize=(14, 12))
    corr_clean = corr.dropna(axis=0, how="any").dropna(axis=1, how="any")

    # 2. Clustermap sur la matrice propre
    sns.clustermap(
        corr_clean,
        method="average",
        cmap="coolwarm",
        vmin=-1, vmax=1,
        figsize=(15, 15)
    )
    plt.show()
    return corr_clean

In [60]:
# MCORR=plot_matrix(df)

In [65]:
# print(MCORR.index)

In [51]:
def detect_corr(M: pd.DataFrame, seuil: float = 0.3) -> dict:
    dicorr = defaultdict(list)
    for col in M.columns:
        dicorr[col] = [
            idx for idx in M.index if idx != col and abs(M.loc[idx, col]) > seuil
        ]
    return dict(dicorr)


In [61]:
# dicorr=detect_corr(plot_matrix(df),seuil=0.5)
# for k in dicorr.keys():
#     print(f"La grandeur {k} est corrélée avec {dicorr[k]}")

In [64]:
# RELU_COL=[k for k in dicorr.keys() if len(dicorr[k])>0]
# sub_df=df[RELU_COL]
#
# sub_dicorr=detect_corr(plot_matrix(sub_df),seuil=0.5)
# for k in sub_dicorr.keys():
#     print(f"La grandeur {k} est corrélée avec {sub_dicorr[k]}")

In [63]:

sns.set_theme(style="whitegrid", context="notebook", palette="Set2")

def plot_distributions(dfs, columns, path="distributions.pdf", min_n=10):
    """dfs : dict {nom_puce: DataFrame}. Un graphique par grandeur, une courbe par puce."""
    with PdfPages(path) as pdf:
        for col in columns:
            # valeurs numériques uniquement, NaN et texte ignorés
            data = {
                chip: pd.to_numeric(df[col], errors="coerce").dropna()
                for chip, df in dfs.items() if col in df.columns
            }
            data = {c: s for c, s in data.items() if len(s) >= min_n}
            if not data:
                continue

            d = pd.concat(
                [pd.DataFrame({"x": np.log10(1 + s), "Puce": f"{c} (n={len(s)})"})
                 for c, s in data.items()]
            )

            fig, ax = plt.subplots(figsize=(7, 4))
            sns.kdeplot(data=d, x="x", hue="Puce", fill=True, alpha=.25,
                        common_norm=False, cut=0, linewidth=2, ax=ax)
            ax.set_title(col, fontweight="bold")
            ax.set_xlabel("log10(1 + valeur)")
            ax.set_ylabel("Densité")
            sns.despine()
            fig.tight_layout()
            pdf.savefig(fig)
            plt.close(fig)

# utilisation : un DataFrame par puce, chargé une seule fois
# dfs = {chip: cleaning(chip) for chip in CHIPS}     # adapte cleaning pour qu'il prenne la puce
# plot_distributions(dfs, COLUMNS)